In [ ]:
import cv2
import numpy as np


# ---------------------------------------------------------
# Read Input Image
# ---------------------------------------------------------
image = cv2.imread("image2.jpg", cv2.IMREAD_GRAYSCALE)

if image is None:
    print("Error: Image could not be loaded.")
    exit()


# ---------------------------------------------------------
# Function for Padding
# ---------------------------------------------------------
def pad_image(image, pad, padding_type):

    if padding_type == "zero":
        padded = cv2.copyMakeBorder(
            image,
            pad, pad, pad, pad,
            cv2.BORDER_CONSTANT,
            value=0
        )

    elif padding_type == "replication":
        padded = cv2.copyMakeBorder(
            image,
            pad, pad, pad, pad,
            cv2.BORDER_REPLICATE
        )

    return padded


# ---------------------------------------------------------
# Averaging Filter (Unweighted)
# ---------------------------------------------------------
def averaging_filter(image, filter_size, padding_type):

    pad = filter_size // 2

    padded = pad_image(image, pad, padding_type)

    output = np.zeros_like(image, dtype=np.float32)

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):

            window = padded[
                i:i + filter_size,
                j:j + filter_size
            ]

            output[i, j] = np.mean(window)

    return np.uint8(np.clip(output, 0, 255))


# ---------------------------------------------------------
# Weighted Filter
# h(x,y) = max(|x|, |y|)
# ---------------------------------------------------------
def weighted_filter(image, filter_size, padding_type):

    pad = filter_size // 2

    padded = pad_image(image, pad, padding_type)

    output = np.zeros_like(image, dtype=np.float32)

    kernel = np.zeros(
        (filter_size, filter_size),
        dtype=np.float32
    )

    for i in range(filter_size):
        for j in range(filter_size):

            x = i - pad
            y = j - pad

            kernel[i, j] = max(abs(x), abs(y))

    # Avoid zero kernel
    if np.sum(kernel) == 0:
        kernel[pad, pad] = 1

    # Normalize kernel
    kernel = kernel / np.sum(kernel)

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):

            window = padded[
                i:i + filter_size,
                j:j + filter_size
            ]

            output[i, j] = np.sum(window * kernel)

    return np.uint8(np.clip(output, 0, 255))


# ---------------------------------------------------------
# Gaussian Filter
# ---------------------------------------------------------
def gaussian_filter(image, filter_size, sigma, padding_type):

    pad = filter_size // 2

    padded = pad_image(image, pad, padding_type)

    kernel = np.zeros(
        (filter_size, filter_size),
        dtype=np.float32
    )

    for i in range(filter_size):
        for j in range(filter_size):

            x = i - pad
            y = j - pad

            kernel[i, j] = np.exp(
                -(x*x + y*y) / (2 * sigma*sigma)
            )

    # Normalize Gaussian kernel
    kernel = kernel / np.sum(kernel)

    output = np.zeros_like(image, dtype=np.float32)

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):

            window = padded[
                i:i + filter_size,
                j:j + filter_size
            ]

            output[i, j] = np.sum(window * kernel)

    return np.uint8(np.clip(output, 0, 255))


# ---------------------------------------------------------
# User Input
# ---------------------------------------------------------

filter_size = int(
    input("Enter filter size (3, 5, 7...): ")
)

if filter_size <= 0 or filter_size % 2 == 0:
    print("Filter size must be a positive odd number.")
    exit()


print("\nChoose Padding Method:")
print("1. Zero Padding")
print("2. Replication Padding")

choice = int(input("Enter your choice: "))

if choice == 1:
    padding_type = "zero"

elif choice == 2:
    padding_type = "replication"

else:
    print("Invalid choice.")
    exit()


sigma = float(
    input("\nEnter sigma for Gaussian filter: ")
)

if sigma <= 0:
    print("Sigma must be greater than 0.")
    exit()


# ---------------------------------------------------------
# Apply Filters
# ---------------------------------------------------------

average_result = averaging_filter(
    image,
    filter_size,
    padding_type
)

weighted_result = weighted_filter(
    image,
    filter_size,
    padding_type
)

gaussian_result = gaussian_filter(
    image,
    filter_size,
    sigma,
    padding_type
)


# ---------------------------------------------------------
# Display Results
# ---------------------------------------------------------

cv2.imshow("Original Image", image)

cv2.imshow(
    "Averaging Filter",
    average_result
)

cv2.imshow(
    "Weighted Filter",
    weighted_result
)

cv2.imshow(
    "Gaussian Filter",
    gaussian_result
)


print("\nAll filters applied successfully!")
print("Press any key to close the windows.")

cv2.waitKey(0)
cv2.destroyAllWindows()

Enter filter size (3, 5, 7...):  3



Choose Padding Method:
1. Zero Padding
2. Replication Padding


Enter your choice:  2

Enter sigma for Gaussian filter:  10



All filters applied successfully!
Press any key to close the windows.
